# MedGemma 1.5 知识蒸馏（QLoRA）- 胸部 X 光报告生成与 RadGraph F1 评估

**方法**: 知识蒸馏（Knowledge Distillation）

**Teacher**: Google MedGemma 1.5 (4B) 原始模型

**Student**: MedGemma 1.5 + QLoRA（4-bit 量化 + LoRA 微调）

**数据集**: MIMIC-CXR (233 samples)

**评估指标**: RadGraph F1 Score

**GPU**: A100 / H100

---

## ⚠️ 环境要求

- **Python**: 3.10-3.12（Colab 默认 3.12 可用）
- **GPU**: A100 或 H100（蒸馏训练需要较多显存，推荐 40GB+）
- **依赖**: transformers 5.x, radgraph, bitsandbytes, peft, trl
- **HuggingFace**: 需获取 MedGemma 访问权限和 token（见 Step 2）
- **Google Drive**: 需上传 `mimic_eval_single_image_final_233.csv`

## 🚨 使用前必读

1. **Step 0**: 检查 Python 版本（3.10-3.12 均可）
2. **Step 2**: ⚠️ **必须先登录 HuggingFace**（MedGemma 是 gated model）
3. **时间预估**: 蒸馏训练需要 2-4 小时（取决于 GPU 和训练步数）

---

## 🎓 知识蒸馏说明

### **逻辑**：
- **Teacher（老师模型）**: 原始 MedGemma 1.5，生成高质量报告
- **Student（学生模型）**: 4-bit 量化 + LoRA 微调的 MedGemma，学习模仿 teacher
- **蒸馏目标**: Student 逐 token 拟合 Teacher 的输出序列（使用 CE 损失）

### **优势**：
- Student 模型更小、更快（4-bit 量化）
- 通过学习 Teacher 的输出，保持较高的生成质量
- LoRA 微调使得训练更高效

---

## 流程

0. 检查 Python 版本
1. 安装依赖（transformers + radgraph + **peft + trl + bitsandbytes**）
2. **登录 HuggingFace**
3. 挂载 Google Drive
4. 下载 MIMIC-CXR 数据集
5. 对齐 233 CSV 的图片路径
6. **使用 Teacher 模型生成目标报告**（或使用 CSV 中已有的）
7. **初始化 Student 模型**（4-bit + LoRA）
8. **蒸馏训练**（Student 学习 Teacher 的输出）
9. **用训练后的 Student 生成报告**
10. **RadGraph F1 评估**

## Step 0: 安装 Python 3.10（必需！）

⚠️ **重要**：Colab 默认是 Python 3.12，但某些依赖需要 3.10 或 3.11。

运行后需要 **Restart Runtime**（Runtime → Restart Runtime）。

## Step 1: 安装依赖

⚠️ **前置条件**：确保已安装 Python 3.10（Step 0）并重启了 Runtime。

In [4]:
import sys

print(f"当前 Python 版本: {sys.version}")

# 检查 Python 版本
py_major = sys.version_info.major
py_minor = sys.version_info.minor

if py_major == 3 and py_minor in [10, 11]:
    print(f"✅ Python 3.{py_minor} 符合要求")
elif py_major == 3 and py_minor == 12:
    print("\\n" + "="*70)
    print("ℹ️ 检测到 Python 3.12")
    print("\\ntransformers 5.x 和 radgraph 都支持 Python 3.12，可以直接使用！")
    print("\\n如果遇到兼容性问题，可以尝试：")
    print("1. 降级 transformers：pip install transformers==4.47.1")
    print("2. 或在 Colab 设置中选择旧版 runtime")
    print("="*70)
    print("\\n✅ 继续使用 Python 3.12（建议先尝试）")
else:
    print(f"\\n⚠️ 未知 Python 版本: {py_major}.{py_minor}")
    print("建议使用 Python 3.10-3.12")

print("\\n跳过 Python 安装，直接进入 Step 1")

当前 Python 版本: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
\n======================================================================
ℹ️ 检测到 Python 3.12
\ntransformers 5.x 和 radgraph 都支持 Python 3.12，可以直接使用！
\n如果遇到兼容性问题，可以尝试：
1. 降级 transformers：pip install transformers==4.47.1
2. 或在 Colab 设置中选择旧版 runtime
\n✅ 继续使用 Python 3.12（建议先尝试）
\n跳过 Python 安装，直接进入 Step 1


## Step 2: 挂载 Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive 已挂载")

Mounted at /content/drive
✅ Google Drive 已挂载


In [5]:
import sys

# 验证 Python 版本
py_version = f"{sys.version_info.major}.{sys.version_info.minor}"
print(f"当前 Python 版本: {py_version}")

if sys.version_info.major == 3 and sys.version_info.minor in [10, 11, 12]:
    print(f"✅ Python {py_version} 兼容")
else:
    print(f"\n⚠️ 警告：当前版本 {py_version} 未测试，推荐 3.10-3.12")

# 安装依赖（蒸馏需要 peft, trl, bitsandbytes）
print("\n正在安装依赖...")
!pip install -U -q transformers
!pip install -q radgraph
!pip install -q bitsandbytes  # 量化必需
!pip install -q peft  # LoRA 微调必需
!pip install -q trl  # 训练工具
print("\n✅ 依赖安装完成（包含 bitsandbytes, peft, trl）")

当前 Python 版本: 3.12
✅ Python 3.12 兼容

正在安装依赖...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 39.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.0/588.0 kB 23.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 540.5/540.5 kB 21.0 MB/s eta 0:00:00

✅ 依赖安装完成（包含 bitsandbytes, peft, trl）


## Step 3: 下载 MIMIC-CXR 数据集

使用 kagglehub 下载 MIMIC-CXR 数据集（~18GB）。首次运行需要 5-10 分钟。

In [6]:
import kagglehub
import os

print("🚀 开始下载 MIMIC-CXR 数据集（~18GB，首次运行需要 5-10 分钟）...")

# kagglehub 会自动缓存，第二次运行会很快
dataset_path = kagglehub.dataset_download("simhadrisadaram/mimic-cxr-dataset")

print(f"✅ 数据集下载完成！")
print(f"📂 存储路径: {dataset_path}")

# 验证文件结构
print("\n--- 文件夹结构预览 ---")
for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level > 1: break  # 只显示前 2 层

🚀 开始下载 MIMIC-CXR 数据集（~18GB，首次运行需要 5-10 分钟）...
Using Colab cache for faster access to the 'mimic-cxr-dataset' dataset.
✅ 数据集下载完成！
📂 存储路径: /kaggle/input/mimic-cxr-dataset

--- 文件夹结构预览 ---
mimic-cxr-dataset/
  official_data_iccv_final/
    files/


## Step 4: 加载 233 CSV 并对齐路径

读取 `mimic_eval_single_image_final_233.csv`，并将 CSV 中的路径对齐到 kagglehub 下载的实际路径。

In [7]:
import pandas as pd
import os

# ✅ 使用 233 CSV
csv_path = "/content/drive/MyDrive/medgamma/mimic_eval_single_image_final_233.csv"

if not os.path.exists(csv_path):
    print(f"❌ 找不到 CSV 文件: {csv_path}")
    print("\n请确保 233 CSV 已上传到 Google Drive 的 medgamma 文件夹中！")
    raise FileNotFoundError(csv_path)

print(f"📂 读取 CSV: {csv_path}")
df = pd.read_csv(csv_path)
print(f"✅ 加载成功！共 {len(df)} 条数据")
print(f"\n列名: {list(df.columns)}")

# ==========================================
# 路径对齐函数
# ==========================================
dataset_root = f"{dataset_path}/official_data_iccv_final"

def fix_image_path(path_in_csv):
    """
    将 233 CSV 中的路径对齐到 kagglehub 下载的实际路径
    CSV 格式: /kaggle/input/mimic-cxr-dataset/official_data_iccv_final/files/...
    实际格式: {dataset_root}/files/...
    """
    if pd.isna(path_in_csv):
        return None

    path_str = str(path_in_csv).strip()

    # 提取从 files/ 开始的相对路径
    if 'files/' in path_str:
        relative_part = path_str.split('files/', 1)[1]
        full_path = os.path.join(dataset_root, 'files', relative_part)
        return full_path if os.path.exists(full_path) else None

    return None

# 应用路径修正
print("\n🔄 正在对齐图片路径...")
if 'Image_Path' in df.columns:
    df['Image_Path'] = df['Image_Path'].apply(fix_image_path)
    # 验证
    valid_count = df['Image_Path'].notna().sum()
    print(f"✅ 路径对齐完成！有效路径: {valid_count}/{len(df)}")

    # 显示第一个有效路径
    first_valid = df['Image_Path'].dropna().iloc[0]
    print(f"示例路径: {first_valid}")
    print(f"文件存在: {os.path.exists(first_valid)}")
else:
    print("❌ CSV 中没有 Image_Path 列！")

📂 读取 CSV: /content/drive/MyDrive/medgamma/mimic_eval_single_image_final_233.csv
✅ 加载成功！共 233 条数据

列名: ['subject_id', 'View', 'Image_Path', 'Ground_Truth', 'Generated_Report']

🔄 正在对齐图片路径...
✅ 路径对齐完成！有效路径: 233/233
示例路径: /kaggle/input/mimic-cxr-dataset/official_data_iccv_final/files/p10/p10075925/s51010496/2d783c8a-492984b7-28aaf571-bfc30156-61ab26f6.jpg
文件存在: True


## Step 6: 使用 Teacher 模型生成目标报告

**说明**: 如果 CSV 中已有 `Generated_Report` 列且质量较好，可以直接使用。

否则需要先加载原始 MedGemma 作为 Teacher 生成目标报告。

In [8]:
from huggingface_hub import login
from google.colab import userdata

try:
    # 使用你已设置的 Secret 名称（zhuxirui11）
    hf_token = userdata.get('zhuxirui11')
    login(token=hf_token)
    print("✅ HuggingFace 登录成功！")
except Exception as e:
    print("❌ 登录失败！")
    print(f"错误信息: {e}")
    print("\n请确认：")
    print("1. 已在 https://huggingface.co/google/medgemma-1.5-4b-it 申请访问权限")
    print("2. 已在 Colab 左侧 🔑 Secrets 中添加 zhuxirui11")
    print("3. 已勾选 'Notebook access'（开关是蓝色的）")
    raise

✅ HuggingFace 登录成功！


In [9]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

# ===== Teacher 模型（原始 MedGemma）=====
model_id = "google/medgemma-1.5-4b-it"

print(f"🤖 正在加载 Teacher 模型: {model_id}...")
from transformers import AutoProcessor, AutoModelForImageTextToText

model_id = "google/medgemma-1.5-4b-it"

print(f"🤖 正在加载模型: {model_id}...")
print("⚠️ 首次加载需要下载 ~8GB 权重，请耐心等待...\n")

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,  # A100 支持 BF16
    device_map="auto",
    trust_remote_code=True
)

print(f"✅ 模型加载成功！")
print(f"Device: {model.device}")
if torch.cuda.is_available():
    mem_gb = torch.cuda.memory_allocated(0) / (1024**3)
    print(f"GPU 显存占用: {mem_gb:.2f} GB")

🤖 正在加载 Teacher 模型: google/medgemma-1.5-4b-it...
🤖 正在加载模型: google/medgemma-1.5-4b-it...
⚠️ 首次加载需要下载 ~8GB 权重，请耐心等待...



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

✅ 模型加载成功！
Device: cuda:0
GPU 显存占用: 8.01 GB


## Step 6: 批量生成报告（233 samples）

对 233 张胸部 X 光图片生成放射学报告。

In [12]:
import re
from PIL import Image
from tqdm import tqdm

def generate_one_report(image_path, view_position):
    """
    为单张图片生成放射学报告
    """
    # 提示词
    prompt_text = (
        f"You are an expert radiologist. Describe this {view_position} view chest X-ray. "
        "Provide a concise report consisting of Findings and Impression. "
        "Focus on the heart, lungs, mediastinum, pleural space, and bones. "
        "Do NOT use bullet points, asterisks, or section headers. "
        "Do NOT include disclaimers or 'AI' warnings. "
        "Output pure medical text only."
    )

    try:
        pil_image = Image.open(image_path).convert("RGB")
    except Exception as e:
        return f"ERROR_IMAGE_LOAD: {e}"

    # 构建输入
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": pil_image},
                {"type": "text", "text": prompt_text}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    # 推理
    with torch.inference_mode():
        generation = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )
        generation = generation[0][input_len:]  # 裁剪掉 prompt

    # ✅ 解码并强力清洗，去除格式标记
    raw_text = processor.decode(generation, skip_special_tokens=True)
    clean_text = raw_text.replace("Findings:", "").replace("Impression:", "")
    clean_text = clean_text.replace("**", "").replace("*", "")  # 去除 markdown
    clean_text = clean_text.replace("###", "").replace("##", "").replace("#", "")
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()

    return clean_text

# ==========================================
# 批量生成
# ==========================================
print(f"\n🚀 开始批量生成报告（共 {len(df)} 条）...\n")

results = []
skipped_count = 0

for idx, row in tqdm(df.iterrows(), total=len(df), desc="生成报告"):
    try:
        # ✅ 使用 Image_Path 列
        img_path = row.get('Image_Path')
        view = row.get('View', 'PA')

        # 验证路径
        if not img_path or not os.path.exists(img_path):
            skipped_count += 1
            continue

        # 生成报告
        generated_report = generate_one_report(img_path, view)

        # 检查错误
        if "ERROR_IMAGE_LOAD" in generated_report:
            skipped_count += 1
            continue

        # ✅ Ground_Truth 列
        gt_col = 'Ground_Truth' if 'Ground_Truth' in df.columns else 'text'
        ground_truth = str(row.get(gt_col, '')).strip()

        # 保存结果
        results.append({
            "subject_id": row.get('subject_id', idx),
            "View": view,
            "Image_Path": img_path,
            "Ground_Truth": ground_truth,
            "Generated_Report": generated_report
        })

    except Exception as e:
        print(f"\n❌ Error at index {idx}: {e}")
        skipped_count += 1
        continue

# 保存结果
df_results = pd.DataFrame(results)
output_path = "/content/drive/MyDrive/medgamma/medgemma_distilled_reports_233.csv"
df_results.to_csv(output_path, index=False)

print(f"\n✅ 报告生成完成！")
print(f"成功生成: {len(results)} 条")
print(f"跳过: {skipped_count} 条")
print(f"结果已保存至: {output_path}")

# 预览前 3 条
print("\n--- 前 3 条报告预览 ---")
for i in range(min(3, len(df_results))):
    print(f"\n[{i+1}] Subject: {df_results.iloc[i]['subject_id']}")
    print(f"Ground Truth: {df_results.iloc[i]['Ground_Truth'][:100]}...")
    print(f"Generated: {df_results.iloc[i]['Generated_Report'][:100]}...")


🚀 开始批量生成报告（共 233 条）...



生成报告: 100%|██████████| 233/233 [14:57<00:00,  3.85s/it]


✅ 报告生成完成！
成功生成: 233 条
跳过: 0 条
结果已保存至: /content/drive/MyDrive/medgamma/medgemma_distilled_reports_233.csv

--- 前 3 条报告预览 ---

[1] Subject: 10075925
Ground Truth: Mild pulmonary vascular congestion with mild to moderate interstitial pulmonary edema are new compar...
Generated: The heart is enlarged. There is mild pulmonary edema. The mediastinum is unremarkable. The pleural s...

[2] Subject: 10174198
Ground Truth: Lungs are clear without consolidation, effusion, or pneumothorax.  The cardiomediastinal silhouette ...
Generated: The heart size is normal. The mediastinal contours are unremarkable. The lungs are clear without foc...

[3] Subject: 10199765
Ground Truth: Subtle patchy opacity along the left heart border on the frontal view, not substantiated on the late...
Generated: The heart size is mildly enlarged. The mediastinal contours are unremarkable. The lungs are clear wi...


## Step 7: 保存 Teacher 输出

将 Teacher 模型生成的报告保存，作为 Student 的学习目标。

In [13]:
# 保存 Teacher 输出到 CSV
teacher_csv_path = "/content/drive/MyDrive/medgamma/teacher_reports_233.csv"
df_results.to_csv(teacher_csv_path, index=False)
print(f"✅ Teacher 报告已保存至: {teacher_csv_path}")

# 释放 Teacher 模型显存
import gc
del model, processor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✅ Teacher 模型已卸载")

✅ Teacher 报告已保存至: /content/drive/MyDrive/medgamma/teacher_reports_233.csv
✅ Teacher 模型已卸载


## Step 8: 初始化 Student 模型（4-bit + LoRA）

**配置**:
- **量化**: 4-bit (bitsandbytes)
- **微调**: LoRA (Parameter-Efficient Fine-Tuning)
- **训练目标**: 学习 Teacher 的输出

In [14]:
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "google/medgemma-1.5-4b-it"

print(f"🤖 正在加载 Student 模型: {model_id} (4-bit + LoRA)...\n")

# 4-bit 量化配置
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# 加载 processor
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

# 加载 Student 模型（4-bit 量化）
student_model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True
)

# 准备模型用于 k-bit 训练
student_model = prepare_model_for_kbit_training(student_model)

# LoRA 配置
lora_config = LoraConfig(
    r=16,  # LoRA rank
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # 应用 LoRA 的模块
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 应用 LoRA
student_model = get_peft_model(student_model, lora_config)
student_model.print_trainable_parameters()

print(f"\n✅ Student 模型（4-bit + LoRA）初始化完成！")
if torch.cuda.is_available():
    mem_gb = torch.cuda.memory_allocated(0) / (1024**3)
    print(f"GPU 显存占用: {mem_gb:.2f} GB")

🤖 正在加载 Student 模型: google/medgemma-1.5-4b-it (4-bit + LoRA)...



Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

trainable params: 6,447,104 || all params: 4,306,526,576 || trainable%: 0.1497

✅ Student 模型（4-bit + LoRA）初始化完成！
GPU 显存占用: 12.33 GB


## Step 9: 蒸馏训练

**训练配置**:
- **Epochs**: 3-5
- **Batch size**: 1-2（取决于显存）
- **Learning rate**: 2e-4
- **Loss**: Cross Entropy（逐 token 拟合 Teacher 输出）

⚠️ **预计时间**: 2-4 小时

In [19]:
# ==========================================
# 方案 2: 固定长度方案（更简单）
# ==========================================

from transformers import Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch
from PIL import Image

class DistillationDataset(Dataset):
    def __init__(self, df, processor, image_root, max_length=512):
        self.df = df
        self.processor = processor
        self.image_root = image_root
        self.max_length = max_length  # ⭐ 固定最大长度

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = row["Image_Path"]
        teacher_output = row["Generated_Report"]

        # 加载图片
        try:
            image = Image.open(image_path).convert("RGB")
        except:
            image = Image.new("RGB", (224, 224), (255, 255, 255))

        # 准备输入
        messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": "Describe this chest X-ray in detail."}]}]
        prompt = self.processor.apply_chat_template(messages, add_generation_prompt=True)

        # ⭐ 关键：使用固定长度 tokenize 输入
        inputs = self.processor(
            text=prompt,
            images=image,
            return_tensors="pt",
            padding="max_length",      # ⭐ 固定 padding
            truncation=True,           # ⭐ 截断
            max_length=self.max_length # ⭐ 固定长度
        )

        # ⭐ 关键：使用相同的固定长度 tokenize teacher output
        teacher_tokens = self.processor.tokenizer(
            teacher_output,
            return_tensors="pt",
            padding="max_length",      # ⭐ 固定 padding
            truncation=True,           # ⭐ 截断
            max_length=self.max_length # ⭐ 相同的固定长度
        )

        # 创建 labels（padding token 设为 -100）
        labels = teacher_tokens["input_ids"].clone()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "pixel_values": inputs["pixel_values"].squeeze(),
            "labels": labels.squeeze()
        }


class CustomDataCollator:
    """为 Gemma3 添加必需的 token_type_ids"""

    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        batch = {
            "input_ids": torch.stack([f["input_ids"] for f in features]),
            "attention_mask": torch.stack([f["attention_mask"] for f in features]),
            "pixel_values": torch.stack([f["pixel_values"] for f in features]),
            "labels": torch.stack([f["labels"] for f in features]),
        }

        # 添加 token_type_ids
        batch_size, seq_length = batch["input_ids"].shape
        batch["token_type_ids"] = torch.zeros((batch_size, seq_length), dtype=torch.long)

        return batch


# ==========================================
# 使用方法
# ==========================================

# 创建数据集（使用固定长度）
train_dataset = DistillationDataset(
    df_results,
    processor,
    dataset_root,
    max_length=512  # ⭐ 可以调整这个值
)

# 创建数据整理器
data_collator = CustomDataCollator(processor)

# 训练参数
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/medgamma/distillation_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
)

# 创建 Trainer
trainer = Trainer(
    model=student_model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

# 开始训练
print("\n🚀 开始蒸馏训练...\n")
trainer.train()

print("\n✅ 蒸馏训练完成！")

# 保存模型
student_model.save_pretrained("/content/drive/MyDrive/medgamma/distilled_student_model")
processor.save_pretrained("/content/drive/MyDrive/medgamma/distilled_student_model")
print("✅ Student 模型已保存")


🚀 开始蒸馏训练...



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,18.554918
20,10.403647
30,6.266368
40,4.504331
50,3.550578
60,3.678167
70,3.609967
80,3.154442
90,2.868221
100,2.636406


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt


✅ 蒸馏训练完成！
✅ Student 模型已保存


## Step 7: RadGraph F1 评估

使用 RadGraph 指标评估生成报告的质量。

**指标说明：**
- **RG_E**: Entity F1（实体匹配）
- **RG_ER**: Entity + Relation F1（实体+关系，论文常用指标）
- **RG_ER_bar**: Complete Match F1（完全匹配）

In [20]:
# ⚠️ RadGraph 兼容性修复（必须先运行这个 cell！）
# 修复 transformers 5.x 与 radgraph 的兼容性问题

from transformers import BertTokenizer
from transformers.tokenization_utils_base import PreTrainedTokenizerBase

fixed_methods = []

# 1. 添加 encode_plus 方法
if not hasattr(BertTokenizer, 'encode_plus'):
    def encode_plus_wrapper(self, text, *args, **kwargs):
        return self(text, *args, **kwargs)
    BertTokenizer.encode_plus = encode_plus_wrapper
    fixed_methods.append('encode_plus')

# 2. 添加 build_inputs_with_special_tokens 方法
if not hasattr(BertTokenizer, 'build_inputs_with_special_tokens'):
    def build_inputs_with_special_tokens_wrapper(self, token_ids_0, token_ids_1=None):
        # BERT: [CLS] + tokens_0 + [SEP] + tokens_1 + [SEP]
        if token_ids_1 is None:
            return [self.cls_token_id] + token_ids_0 + [self.sep_token_id]
        return [self.cls_token_id] + token_ids_0 + [self.sep_token_id] + token_ids_1 + [self.sep_token_id]
    BertTokenizer.build_inputs_with_special_tokens = build_inputs_with_special_tokens_wrapper
    fixed_methods.append('build_inputs_with_special_tokens')

# 3. 添加 get_special_tokens_mask 方法（可能也会需要）
if not hasattr(BertTokenizer, 'get_special_tokens_mask'):
    def get_special_tokens_mask_wrapper(
        self, token_ids_0, token_ids_1=None, already_has_special_tokens=False
    ):
        if already_has_special_tokens:
            return super(BertTokenizer, self).get_special_tokens_mask(
                token_ids_0=token_ids_0,
                token_ids_1=token_ids_1,
                already_has_special_tokens=True,
            )
        if token_ids_1 is None:
            return [1] + ([0] * len(token_ids_0)) + [1]
        return [1] + ([0] * len(token_ids_0)) + [1] + ([0] * len(token_ids_1)) + [1]
    BertTokenizer.get_special_tokens_mask = get_special_tokens_mask_wrapper
    fixed_methods.append('get_special_tokens_mask')

if fixed_methods:
    print(f"✅ RadGraph 兼容性修复已应用: {', '.join(fixed_methods)}")
else:
    print("✅ BertTokenizer 已有所有必需方法")

✅ RadGraph 兼容性修复已应用: encode_plus, build_inputs_with_special_tokens


In [25]:
import torch

print("=" * 60)
print("📊 GPU 显存使用情况")
print("=" * 60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    current_gb = torch.cuda.memory_allocated(0) / (1024**3)
    peak_gb = torch.cuda.max_memory_allocated(0) / (1024**3)

    print(f"\nGPU: {gpu_name}")
    print(f"总显存: {total_gb:.2f} GB")
    print(f"当前使用: {current_gb:.2f} GB")
    print(f"峰值使用: {peak_gb:.2f} GB  ← 这是模型加载时的最大显存")
    print(f"剩余: {total_gb - current_gb:.2f} GB")
else:
    print("CPU 模式")

print("=" * 60)


# ==========================================
# 第 2 步: 准备评估数据
# ==========================================

import pandas as pd

print("\n📁 准备评估数据...")

# 选择一个方法加载数据:

# 方法 A: 从保存的文件加载
# df_sub = pd.read_csv("/content/drive/MyDrive/medgamma/generated_reports.csv")

# 方法 B: 如果你有 df_results 变量
df_sub = df_results.copy()

# 方法 C: 如果你有其他变量名
# df_sub = your_dataframe_name.copy()

# 验证数据
print(f"✅ 数据加载成功: {len(df_sub)} 条")
print(f"列名: {df_sub.columns.tolist()}")


# ==========================================
# 第 3 步: RadGraph 评估
# ==========================================

from radgraph import F1RadGraph

print("\n🔍 开始 RadGraph F1 评估...")

# 准备数据
refs = df_sub["Ground_Truth"].fillna("").tolist()
hyps = df_sub["Generated_Report"].fillna("").tolist()

# 过滤空报告
valid_pairs = [(h, r) for h, r in zip(hyps, refs) if h and r and len(h.strip()) > 0]
print(f"有效样本: {len(valid_pairs)}")

hyps_clean, refs_clean = zip(*valid_pairs)

# 计算
print("计算中...")
f1radgraph = F1RadGraph(reward_level="all")
results = f1radgraph(hyps=list(hyps_clean), refs=list(refs_clean))

# 结果
avg_scores = results[0]
simple_f1 = float(avg_scores[0])
partial_f1 = float(avg_scores[1])
complete_f1 = float(avg_scores[2])

# 显示
print("\n" + "=" * 60)
print("RadGraph F1 评估结果")
print("-" * 60)
print(f"RG_E:      {complete_f1*100:.2f}%")
print(f"RG_ER:     {partial_f1*100:.2f}%  ← 论文常用")
print(f"RG_ER_bar: {simple_f1*100:.2f}%")
print(f"\nGPU 峰值: {peak_gb:.2f} GB")
print("=" * 60)

print("\n✅ 完成!")

📊 GPU 显存使用情况

GPU: NVIDIA A100-SXM4-80GB
总显存: 79.25 GB
当前使用: 12.41 GB
峰值使用: 28.35 GB  ← 这是模型加载时的最大显存
剩余: 66.84 GB

📁 准备评估数据...
✅ 数据加载成功: 233 条
列名: ['subject_id', 'View', 'Image_Path', 'Ground_Truth', 'Generated_Report']

🔍 开始 RadGraph F1 评估...
有效样本: 233
计算中...
Using device: cuda:0
model_type not provided, defaulting to radgraph-xl


radgraph-xl.tar.gz:   0%|          | 0.00/416M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/radgraph/radgraph.py:105: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=model_dir)


config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/bert/tokenization_bert.py:104: DeprecationWarning: Deprecated in 0.9.0: WordPiece.__init__ will not create from files anymore, try `WordPiece.from_file` instead
  self._tokenizer = Tokenizer(WordPiece(self._vocab, unk_token=str(unk_token)))



RadGraph F1 评估结果
------------------------------------------------------------
RG_E:      27.05%
RG_ER:     34.65%  ← 论文常用
RG_ER_bar: 36.40%

GPU 峰值: 28.35 GB

✅ 完成!


In [ ]:
# ==========================================
# Step 8.1: 使用 Student 模型生成目标报告
# ==========================================

import re
from PIL import Image
from tqdm import tqdm
import pandas as pd
import torch
import os

# 确保 processor 处于正确状态 (Student model's processor)
# 在 Step 8 初始化 Student 模型时，processor 已经被加载
# 我们需要确保 generate_one_report 函数能够访问到 student_model 和 processor

# 为了复用 generate_one_report 函数，我们临时将 student_model 赋值给 model
# 这是一个简化的方式，确保后续代码使用 Student 模型进行推理
original_model = None
if 'model' in globals():
    original_model = globals()['model']
globals()['model'] = student_model

print(f"\n🚀 开始批量生成报告（共 {len(df)} 条）...\n")

student_results = []
skipped_count_student = 0

# 复用 Step 6 中的 generate_one_report 函数
def generate_one_report_for_student(image_path, view_position):
    """
    为单张图片生成放射学报告 (使用当前活跃的 model/processor)
    """
    prompt_text = (
        f"You are an expert radiologist. Describe this {view_position} view chest X-ray. "
        "Provide a concise report consisting of Findings and Impression. "
        "Focus on the heart, lungs, mediastinum, pleural space, and bones. "
        "Do NOT use bullet points, asterisks, or section headers. "
        "Do NOT include disclaimers or 'AI' warnings. "
        "Output pure medical text only."
    )

    try:
        pil_image = Image.open(image_path).convert("RGB")
    except Exception as e:
        return f"ERROR_IMAGE_LOAD: {e}"

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": pil_image},
                {"type": "text", "text": prompt_text}
            ]
        }
    ]

    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )
        generation = generation[0][input_len:]

    raw_text = processor.decode(generation, skip_special_tokens=True)
    clean_text = raw_text.replace("Findings:", "").replace("Impression:", "")
    clean_text = clean_text.replace("**", "").replace("*", "")
    clean_text = clean_text.replace("###", "").replace("##", "").replace("#", "")
    clean_text = re.sub(r'\s+', ' ', clean_text).strip()

    return clean_text

for idx, row in tqdm(df.iterrows(), total=len(df), desc="生成 Student 报告"):
    try:
        img_path = row.get('Image_Path')
        view = row.get('View', 'PA')

        if not img_path or not os.path.exists(img_path):
            skipped_count_student += 1
            continue

        generated_report_student = generate_one_report_for_student(img_path, view)

        if "ERROR_IMAGE_LOAD" in generated_report_student:
            skipped_count_student += 1
            continue

        gt_col = 'Ground_Truth' if 'Ground_Truth' in df.columns else 'text'
        ground_truth = str(row.get(gt_col, '')).strip()

        student_results.append({
            "subject_id": row.get('subject_id', idx),
            "View": view,
            "Image_Path": img_path,
            "Ground_Truth": ground_truth,
            "Generated_Report": generated_report_student
        })

    except Exception as e:
        print(f"\n❌ Error at index {idx} during student generation: {e}")
        skipped_count_student += 1
        continue

df_student_generated_reports = pd.DataFrame(student_results)
output_path_student = "/content/drive/MyDrive/medgamma/student_distilled_reports_233.csv"
df_student_generated_reports.to_csv(output_path_student, index=False)

print(f"\n✅ Student 报告生成完成！")
print(f"成功生成: {len(student_results)} 条")
print(f"跳过: {skipped_count_student} 条")
print(f"结果已保存至: {output_path_student}")

# 恢复原始 model 变量 (如果存在)
if original_model is not None:
    globals()['model'] = original_model

# 清理显存
del student_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("✅ Student 模型已卸载")


🚀 开始批量生成报告（共 233 条）...



生成 Student 报告:   0%|          | 0/233 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
Caching is incompatible with gradient checkpointing in Gemma3DecoderLayer. Setting `past_key_values=None`.
生成 Student 报告:  47%|████▋     | 109/233 [1:26:38<1:38:35, 47.70s/it]Setting `pad_token_id` to `eos_token_id`:1 fo

In [27]:
import torch
import pandas as pd
from radgraph import F1RadGraph

print("=" * 60)
print("📊 GPU 显存使用情况")
print("=" * 60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    current_gb = torch.cuda.memory_allocated(0) / (1024**3)
    peak_gb = torch.cuda.max_memory_allocated(0) / (1024**3)

    print(f"\nGPU: {gpu_name}")
    print(f"总显存: {total_gb:.2f} GB")
    print(f"当前使用: {current_gb:.2f} GB")
    print(f"峰值使用: {peak_gb:.2f} GB  ← 这是模型加载时的最大显存")
    print(f"剩余: {total_gb - current_gb:.2f} GB")
else:
    print("CPU 模式")

print("=" * 60)


# ==========================================
# 第 2 步: 准备评估数据 (使用 Student 模型生成的报告)
# ==========================================

print("\n📁 准备评估数据 (来自 Student 模型生成的报告)...")

# 使用新生成的 df_student_generated_reports 作为评估数据
if 'df_student_generated_reports' in globals():
    df_sub = df_student_generated_reports.copy()
    print(f"✅ Student 模型报告加载成功: {len(df_sub)} 条")
    print(f"列名: {df_sub.columns.tolist()}")
else:
    raise NameError("df_student_generated_reports 未定义。请确保运行了 Student 模型报告生成步骤。")


# ==========================================
# 第 3 步: RadGraph 评估
# ==========================================

print("\n🔍 开始 RadGraph F1 评估 (Student 模型)...")

# 准备数据
refs = df_sub["Ground_Truth"].fillna("").tolist()
hyps = df_sub["Generated_Report"].fillna("").tolist()

# 过滤空报告
valid_pairs = [(h, r) for h, r in zip(hyps, refs) if h and r and len(h.strip()) > 0]
print(f"有效样本: {len(valid_pairs)}")

hyps_clean, refs_clean = zip(*valid_pairs)

# 计算
print("计算中...")
f1radgraph = F1RadGraph(reward_level="all")
results = f1radgraph(hyps=list(hyps_clean), refs=list(refs_clean))

# 结果
avg_scores = results[0]
simple_f1 = float(avg_scores[0])
partial_f1 = float(avg_scores[1])
complete_f1 = float(avg_scores[2])

# 显示
print("\n" + "=" * 60)
print("RadGraph F1 评估结果 (Student 模型)")
print("-" * 60)
print(f"RG_E:      {complete_f1*100:.2f}%")
print(f"RG_ER:     {partial_f1*100:.2f}%  ← 论文常用")
print(f"RG_ER_bar: {simple_f1*100:.2f}%")
print(f"\nGPU 峰值: {peak_gb:.2f} GB")
print("=" * 60)

print("\n✅ 完成!")


📊 GPU 显存使用情况

GPU: NVIDIA A100-SXM4-80GB
总显存: 79.25 GB
当前使用: 12.85 GB
峰值使用: 28.35 GB  ← 这是模型加载时的最大显存
剩余: 66.40 GB

📁 准备评估数据 (来自 Student 模型生成的报告)...


NameError: df_student_generated_reports 未定义。请确保运行了 Student 模型报告生成步骤。

## 附录：RadGraph Patch（如遇到兼容性问题）

如果 RadGraph 报错 `encode_plus` 相关错误，运行下面的 patch：

In [ ]:
import json

del model, processor
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

df_sub.to_csv("/kaggle/working/w4a4_medgemma_results.csv", index=False)
with open("/kaggle/working/w4a4_scores.json", "w") as f:
    json.dump({"scores": W4A4_SCORES, "gpu_gb": W4A4_GPU_GB}, f)

print("✅ W4A4 结果已保存至 /kaggle/working/")
print("\n下一步：运行 03_w4a8 或 04_compare")